# Thermo-optic phase shifter: TiN heater over a silicon strip

How much heater power shifts the phase of a silicon waveguide by π, and how fast does the heater switch? This notebook
answers both for a published device. It models the TiN-over-strip heater of Jacques et al., *Opt. Express* **27**,
10456 (2019), [doi:10.1364/OE.27.010456](https://doi.org/10.1364/OE.27.010456), on 220 nm SOI. It then compares the
results with that paper's measurements.

[`gsim.thermal`](../api/thermal.md) solves steady and transient heat conduction in a 2D cross-section of an infinitely
long heater. It then re-solves the waveguide's optical mode with the heated refractive index. This gives P_π, the
switching time, the heater resistance and V_π, and the crosstalk onto a neighbouring waveguide.

**Requirements:** the `femwell` extra (`pip install "gsim[femwell]"`). Everything runs locally in a few minutes, with no
cloud job.

**Units:** lengths in µm, heater power in mW (total over the heater length), temperature as a rise above ambient in K,
time in µs. Material properties are in SI units.

## Inputs: what comes from the paper and what we assume

The cross-section follows the two open reproductions of the paper: femwell's `metal_heater_phase_shifter` example and
Tidy3D's `TransientThermoOpticShifter` notebook. We could not read the paper's full text, so the thermal constants are
those both reproductions attribute to the paper's Table 3.

| Input | Value | Source |
| --- | --- | --- |
| Si strip | 0.50 × 0.22 µm | paper, via both reproductions |
| Buried oxide (BOX) | 2.0 µm | paper, via both reproductions |
| Oxide gap, Si top to heater bottom | 2.0 µm | paper, via both reproductions |
| TiN heater | 2.0 µm wide, **0.14 µm** thick, 320 µm long | width and length: paper; thickness: femwell (Tidy3D uses 0.20 µm) |
| Oxide above the heater | 0.5 µm | assumption (femwell ≈ 0.44 µm, Tidy3D 0.08 µm then air) |
| k: Si strip / Si substrate / SiO2 / TiN (W/m/K) | **90** / 148 / 1.38 / 28 | paper Table 3; 90 is the thin-film value for 220 nm Si |
| dn/dT: Si / SiO2 (1/K) | 1.86e-4 / 1e-5 | femwell; gsim's built-in Si value is the same |
| TiN resistivity | 43 µΩ·cm (σ = 2.3e6 S/m) | femwell; it sets R and V_π only |
| Si substrate | 10 µm modelled, bottom held at ambient | assumption; see the convergence section |
| Top and sides | adiabatic | assumption; free convection (h ≈ 10 W/m²/K) is negligible |

**Heater thickness.** The two reproductions disagree: 0.14 µm (femwell) and 0.20 µm (Tidy3D). We use 0.14 µm because
femwell is the only reproduction that also gives the TiN resistivity. That makes it a complete input set for R and V_π,
and lets us check gsim against femwell with the same inputs. The convergence table shows that 0.20 µm changes P_π by
about 1 %. The heater sits in a poor thermal conductor, so its thickness hardly matters for the heat flow.

In [ ]:
import math
import warnings

import gdsfactory as gf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gsim.common.stack import MATERIALS_DB, Layer, LayerStack
from gsim.thermal import ThermalOptions, ThermalSim

gf.gpdk.PDK.activate()

WAVELENGTH = 1.55  # um
LENGTH = 320.0  # heater length, um
CORE_W, CORE_T = 0.50, 0.22  # Si strip, um
BOX = 2.0  # buried oxide, um
GAP = 2.0  # Si top to heater bottom, um
HEATER_W, HEATER_T = 2.0, 0.14  # TiN heater, um
CLAD_ABOVE = 0.5  # oxide above the heater, um
SUBSTRATE = 10.0  # modelled Si substrate below the BOX, um

# Measured (Jacques et al. 2019) and reproduced reference values
P_PI_MEASURED = 21.4  # mW
V_PI_MEASURED = 2.33  # V (via Krause & Malka, Sensors 23, 8587 (2023), Table 1)
TAU_MEASURED = 5.6  # us
P_PI_FEMWELL, P_PI_TIDY3D = 22.6, 23.9  # mW

## Geometry

The device is a straight Si strip under a TiN strip. gdsfactory's `strip_heater_metal` cross-section draws both: the
waveguide on `WG` (1/0) and the heater on `HEATER` (47/0). We cut it at mid-length, `x = 160 µm`, so the in-plane axis of
the cross-section is `y`.

In [ ]:
def heater_over_strip(neighbour_gap=None):
    # TiN heater over a Si strip, optionally with a second strip at neighbour_gap um.
    xs = gf.cross_section.strip_heater_metal(width=CORE_W, heater_width=HEATER_W)
    c = gf.Component()
    c << gf.components.straight(length=LENGTH, cross_section=xs)
    if neighbour_gap is not None:
        victim = c << gf.components.straight(length=LENGTH, width=CORE_W)
        victim.dmovey(neighbour_gap)
    return c


component = heater_over_strip()
component.plot()

## Layer stack

The stack gives each layer its height and material. The Si strip and the Si substrate are both silicon, but they conduct
heat differently. A 220 nm film conducts about 90 W/m/K, and bulk silicon about 148 W/m/K. `ThermalSim.set_material`
overrides a property for every layer made of a given material. So the strip gets its own material name, `si_film`, with
silicon's optical record, and the override then reaches only the strip.

In [ ]:
heater_z0 = CORE_T + GAP
clad_top = heater_z0 + HEATER_T + CLAD_ABOVE


def jacques_stack(heater_t=HEATER_T, substrate=SUBSTRATE):
    # Paper cross-section: z = 0 is the BOX top, the Si substrate is the heat sink.
    top = heater_z0 + heater_t + CLAD_ABOVE
    return LayerStack(
        pdk_name="jacques2019",
        layers={
            "core": Layer(
                name="core",
                gds_layer=(1, 0),
                zmin=0.0,
                zmax=CORE_T,
                thickness=CORE_T,
                material="si_film",
                layer_type="dielectric",
            ),
            "heater": Layer(
                name="heater",
                gds_layer=(47, 0),
                zmin=heater_z0,
                zmax=heater_z0 + heater_t,
                thickness=heater_t,
                material="TiN",
                layer_type="conductor",
            ),
        },
        # Si film: silicon's optical and thermal record, under its own name.
        materials={"si_film": MATERIALS_DB["silicon"].to_dict()},
        dielectrics=[
            {
                "name": "substrate",
                "zmin": -BOX - substrate,
                "zmax": -BOX,
                "material": "silicon",
            },
            {"name": "box", "zmin": -BOX, "zmax": 0.0, "material": "SiO2"},
            {"name": "clad", "zmin": 0.0, "zmax": top, "material": "SiO2"},
        ],
    )

## Simulation setup

The **Window** is the in-plane extent of the cross-section. It spans ±30 µm, more than six times the 4.86 µm of oxide
between the heater and the substrate, so its adiabatic sides hardly affect the result. The **Modelled depth** is the
10 µm of substrate whose bottom is held at ambient. The first waveguide added is the phase shifter, and any later one is
a crosstalk victim.

In [ ]:
def make_sim(
    window=(-30.0, 30.0), depth=SUBSTRATE, heater_t=HEATER_T, neighbour_gap=None
):
    sim = ThermalSim()
    sim.set_geometry(heater_over_strip(neighbour_gap))
    sim.set_stack(jacques_stack(heater_t=heater_t, substrate=max(depth, SUBSTRATE)))
    sim.set_cross_section("x=160", window=window, depth=depth)
    sim.set_heater("heater", length=LENGTH)
    # Thin-film Si conductivity, femwell's TiN resistivity, paper's oxide dn/dT
    sim.set_material("si_film", thermal_conductivity=90.0)
    sim.set_material("TiN", conductivity=2.3e6)
    sim.set_material("SiO2", dn_dT=1e-5)
    sim.add_waveguide("arm", position=0.0)
    if neighbour_gap is not None:
        sim.add_waveguide("neighbour", position=neighbour_gap)
    return sim


sim = make_sim()

## Steady state: P_π and the temperature field

`solve` runs one heat solve. The model is linear in power, so that one solve gives the temperature rise at every power.
It then re-solves the waveguide's mode with the heated index to get the effective-index shift per milliwatt. P_π is
the power at which that shift adds up to a phase of π over the 320 µm heater.

In [ ]:
result = sim.solve(wavelength=WAVELENGTH)
print(f"P_pi = {result.p_pi:.2f} mW, efficiency = {result.efficiency:.4f} rad/mW")
print(f"Core rise per mW = {result.thermal_resistance['arm']:.3f} K/mW")

In [ ]:
tmap = result.temperature(power=result.p_pi)

fig, (ax_map, ax_cut) = plt.subplots(
    1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [2, 1]}
)
tmap.plot(ax=ax_map)
ax_map.set_xlim(-8, 8)
ax_map.set_ylim(-4, clad_top)

z = np.linspace(-BOX - SUBSTRATE + 0.01, clad_top - 0.01, 400)
ax_cut.plot(tmap.at(0.0, z), z)
for z_line, label in [(0.0, "BOX top"), (-BOX, "substrate top"), (heater_z0, "heater")]:
    ax_cut.axhline(z_line, color="0.6", lw=0.8, ls=":")
    ax_cut.text(0.5, z_line + 0.15, label, fontsize=8, color="0.4")
ax_cut.set_xlabel("Temperature rise at y = 0 (K)")
ax_cut.set_ylabel("z (µm)")
ax_cut.set_title("Vertical cut through the heater")
fig.tight_layout()

for z_probe, label in [
    (heater_z0 + HEATER_T / 2, "heater centre"),
    (CORE_T / 2, "core centre"),
    (-BOX, "substrate top"),
]:
    print(
        f"rise at {label:14s} (z = {z_probe:5.2f} um): {tmap.at(0.0, z_probe):6.2f} K"
    )

Almost all of the temperature drop is across the oxide. The silicon substrate, 100 times more conductive, stays within
a few tenths of a kelvin of ambient. That is why P_π depends on the oxide's conductivity and thickness much more than on
the heater or the silicon.

### P_π against measurement and the reproductions

As a sanity check we also use a closed form. It treats the heater as a strip of line sources in uniform oxide, H = 4.86
µm thick, with an adiabatic top and an ideal heat sink below (method of images). It evaluates the rise at the core
centre and takes ΔT_π = λ / (2 L dn/dT_Si). That closed form ignores the Si core and the mode's spread, so it agrees only
to about 10 %.

In [ ]:
def strip_over_sink_rise(H, depth_heater, depth_point, width, k, n=401):
    # Rise (K per W/m) at a point on the axis under a strip heater in a uniform layer
    # with an adiabatic top and an ideal sink at depth H (method of images).
    def line(x, y, y0):
        c = np.cosh(np.pi * x / (2 * H))
        return np.log(
            (c + np.cos(np.pi * (y + y0) / (2 * H)))
            / (c - np.cos(np.pi * (y - y0) / (2 * H)))
        )

    xs = np.linspace(-width / 2, width / 2, n)
    g = line(xs, depth_point, depth_heater) + line(xs, depth_point, -depth_heater)
    return float(np.mean(g)) / (4 * np.pi * k)


H_OXIDE = clad_top + BOX  # adiabatic top to the substrate, um
rise_per_w_per_m = strip_over_sink_rise(
    H_OXIDE, CLAD_ABOVE + HEATER_T / 2, clad_top - CORE_T / 2, HEATER_W, k=1.38
)
dT_pi = WAVELENGTH / (2 * LENGTH * 1.86e-4)  # K
p_pi_closed_form = dT_pi / rise_per_w_per_m * LENGTH * 1e-6 * 1e3  # mW

comparison = pd.DataFrame(
    {
        "P_pi (mW)": [P_PI_MEASURED, P_PI_FEMWELL, P_PI_TIDY3D, p_pi_closed_form],
        "gsim vs it": [
            f"{result.p_pi / p - 1:+.1%}"
            for p in (P_PI_MEASURED, P_PI_FEMWELL, P_PI_TIDY3D, p_pi_closed_form)
        ],
    },
    index=[
        "measured (Jacques 2019), ±15 % target",
        "femwell reproduction",
        "Tidy3D reproduction",
        "closed form, uniform oxide",
    ],
)
print(f"gsim P_pi = {result.p_pi:.2f} mW")
comparison.round(2)

The model lands inside the ±15 % band around the measured 21.4 mW, a few percent high. It agrees with femwell to about
1 %, which is expected: femwell uses a 0.5 µm substrate where we model 10 µm, and its heater is otherwise the same. The
Tidy3D figure is likely the paper's own simulated value and is also close. The remaining gap to the measurement is about
the size of the uncertainty in thin-film oxide conductivity (±10 %). To first order P_π is proportional to that
conductivity.

## Heater resistance and V_π: a discrepancy we cannot resolve

The resistance follows from the drawn heater section and the TiN resistivity: R = ρL / (w t). The measured V_π = 2.33 V
and P_π = 21.4 mW imply R = V_π² / P_π ≈ 254 Ω. With the femwell resistivity (43 µΩ·cm) and 2 × 0.14 µm, the drawn
heater gives about twice that. We report this mismatch instead of tuning it away.

In [ ]:
r_implied = V_PI_MEASURED**2 / (P_PI_MEASURED * 1e-3)
print(f"Model:    R = {result.resistance:.0f} ohm, V_pi = {result.v_pi:.2f} V")
print(
    f"Measured: R = {r_implied:.0f} ohm (from V_pi^2 / P_pi), V_pi = {V_PI_MEASURED} V"
)
for t in (0.14, 0.20):
    rho = r_implied * HEATER_W * 1e-6 * t * 1e-6 / (LENGTH * 1e-6)  # ohm m
    print(
        f"  resistivity that would give {r_implied:.0f} ohm with a 2 x {t} um heater: {rho * 1e8:.0f} uOhm cm"
    )

At 0.14 µm the measured resistance needs about 22 µΩ·cm. That is below the roughly 30 µΩ·cm of the best epitaxial TiN,
so it is implausible for a sputtered film. At 0.20 µm it needs about 32 µΩ·cm, which is plausible. So the measured R
favours the thicker heater, or a drawn heater wider than 2 µm: the review literature lists a 7.5 µm footprint. Neither
choice moves P_π by more than about 1 %. V_π scales as √R, so the model's V_π is about 1.4 times the measured value. Use
your own foundry's sheet resistance (`sim.set_material("TiN", conductivity=...)`) before trusting V_π.

## Switching time: the step response

`solve_transient` switches the heater on at t = 0 and follows the phase until it settles. It reports two times:

- `tau`, the 10–90 % rise time;
- `tau_1e`, the time to reach 1 − 1/e (63 %) of the final phase.

The paper quotes τ = 5.6 µs. We could not check which definition it uses. The Tidy3D reproduction compares 1/e levels.

In [ ]:
step = sim.solve_transient(power=result.p_pi, duration=100.0, wavelength=WAVELENGTH)

ax = step.plot()
for frac, style in [(0.1, ":"), (0.9, ":"), (1 - 1 / math.e, "--")]:
    ax.axhline(frac * step.steady_phase, color="0.7", lw=0.8, ls=style)
ax.axvline(step.tau_1e, color="C1", ls="--", label=f"tau_1e = {step.tau_1e:.1f} µs")
ax.axvline(TAU_MEASURED, color="C3", ls="-.", label=f"measured tau = {TAU_MEASURED} µs")
ax.legend()
print(
    f"tau (10-90 %) = {step.tau:.1f} us, tau_1e = {step.tau_1e:.1f} us, measured {TAU_MEASURED} us (+/-30 % target)"
)
print(
    f"tau_1e / measured = {step.tau_1e / TAU_MEASURED:.2f}, tau / measured = {step.tau / TAU_MEASURED:.2f}"
)

In [ ]:
step_fast = sim.solve_transient(
    power=result.p_pi, duration=100.0, options=ThermalOptions(precision="fast")
)
print(f"fast: tau = {step_fast.tau:.2f} us, tau_1e = {step_fast.tau_1e:.2f} us")
print(f"default: tau = {step.tau:.2f} us, tau_1e = {step.tau_1e:.2f} us")
result.report(step)

**τ misses the target.** The 1/e time comes out about 50 % above the measured 5.6 µs, outside the ±30 % band. The
10–90 % time is more than three times the measurement. We did not tune any input to close the gap. The numerics are not
the cause:

- **The simple physics agrees with the model.** Heat diffusing through the 4.86 µm of oxide has a slowest time
  constant of 4ρc_pH² / (π²k) ≈ 11 µs (1D, adiabatic top, sink below). The model's 1/e time sits under that, as it
  should, because the waveguide is closer to the heater than the sink is and heat also spreads sideways.
- **The response has one dominant time constant.** The ratio of the 10–90 % time to the 1/e time is about 2.15. A pure
  single exponential gives 2.2.
- **The time step is converged.** The cell above repeats the step with `precision="fast"`, which uses half the time
  steps and a coarser mesh.

Likely reasons for the gap, roughly in order:

1. **Definition.** If the paper's 5.6 µs is a fit to the early part of the response, or comes from a 3 dB bandwidth
   with a different single-pole conversion, it reads short of both of our times. We could not verify this.
2. **Thin-film thermal properties.** τ scales with ρc_p/k of the oxide. Deposited oxides and thin films are not bulk
   fused silica. Matching 5.6 µs would need ρc_p/k about 35 % lower, or a thinner oxide.
3. **Geometry of the "optimum" design.** The measured device may differ from the 2 µm-gap section modelled here, for
   example by its oxide thickness or by trenches.
4. **2D.** The real heater has ends and contacts, and the substrate spreads heat in three dimensions.

## Convergence and sensitivity

Each row below changes one model parameter and re-solves. To save time, the Window and depth rows use
`precision="fast"` and are compared with a fast baseline. The precision rows are compared with the default solve above.

In [ ]:
fast = ThermalOptions(precision="fast")
base_fast = make_sim().solve(wavelength=WAVELENGTH, options=fast).p_pi

cases = {
    "Window ±45 µm (×1.5)": (dict(window=(-45.0, 45.0)), fast, base_fast),
    "Modelled depth 5 µm": (dict(depth=5.0), fast, base_fast),
    "Modelled depth 20 µm": (dict(depth=20.0), fast, base_fast),
    "Modelled depth 50 µm": (dict(depth=50.0), fast, base_fast),
    "Heater 0.20 µm thick": (dict(heater_t=0.20), fast, base_fast),
    'precision="fast"': (dict(), fast, result.p_pi),
    'precision="accurate"': (dict(), ThermalOptions(precision="accurate"), result.p_pi),
}
rows = []
for label, (kwargs, options, reference) in cases.items():
    p_pi = make_sim(**kwargs).solve(wavelength=WAVELENGTH, options=options).p_pi
    rows.append(
        {
            "change": label,
            "P_pi (mW)": round(p_pi, 2),
            "vs reference": f"{p_pi / reference - 1:+.2%}",
        }
    )
pd.DataFrame(rows).set_index("change")

- **Window and mesh are converged.** Growing the Window by half, or refining the mesh, changes P_π by well under 1 %.
- **Modelled depth is not a convergence parameter in 2D.** The more substrate we model, the hotter the device runs, so
  P_π drops slowly. A 2D section has no steady state without a sink at a finite distance, and heat that reaches the
  substrate spreads only sideways. In the real chip it also spreads along the 320 µm heater, which a 2D section cannot
  capture. We keep 10 µm, close to Tidy3D's 6 µm. Every depth in the table still sits inside the ±15 % band around the
  measurement, so this choice does not decide the comparison.

## Crosstalk onto a neighbouring waveguide

A second strip at a distance `gap` from the phase shifter picks up some phase when the heater drives P_π. Above an
ideal heat sink, the closed form for a heater on a layer of thickness H predicts that the far-field temperature decays
as exp(−πx / 2H). That is a decay length of 2H/π ≈ 3.1 µm for H = 4.86 µm.

We solve the device twice:

- with the 10 µm Si substrate;
- with an almost ideal sink, a 50 nm substrate held at ambient right under the BOX.

That separates the oxide's own decay from heat that travels through the substrate.

In [ ]:
gaps = [5.0, 10.0, 15.0, 20.0, 30.0]
crosstalk = {"10 µm Si substrate": [], "ideal sink under the BOX": []}
for gap in gaps:
    for label, depth in [
        ("10 µm Si substrate", SUBSTRATE),
        ("ideal sink under the BOX", 0.05),
    ]:
        s = make_sim(window=(-30.0, gap + 30.0), depth=depth, neighbour_gap=gap)
        r = s.solve(wavelength=WAVELENGTH, options=fast)
        crosstalk[label].append(r.crosstalk["neighbour"])

decay_length = 2 * H_OXIDE / math.pi
anchor = crosstalk["ideal sink under the BOX"][1]  # match the closed form at 10 um
analytic = anchor * np.exp(-(np.array(gaps) - gaps[1]) / decay_length)

fig, ax = plt.subplots()
for label, values in crosstalk.items():
    ax.semilogy(gaps, values, "o-", label=label)
ax.semilogy(
    gaps, analytic, "k--", label=f"exp(-x / {decay_length:.2f} µm), closed form"
)
ax.set_xlabel("Gap between waveguide centres (µm)")
ax.set_ylabel("Neighbour phase at P_π (rad)")
ax.set_title("Thermal crosstalk")
ax.legend()

pd.DataFrame(crosstalk, index=pd.Index(gaps, name="gap (µm)")).assign(
    **{"closed form, ideal sink": analytic}
).map(lambda v: f"{v:.2e}")

With an ideal sink under the BOX, the crosstalk follows the closed-form exponential decay over more than three decades.
That is the check on the solver. With the real 10 µm Si substrate the decay slows beyond about 10 µm. Heat that reaches
the substrate spreads sideways and warms the oxide under the neighbour. At 20 µm the crosstalk is about 1.7 times the
ideal-sink value, and at 30 µm about 5 times, so the substrate path dominates the far field. Measured crosstalk on SOI shows the same slow,
power-law-like tail: Harris et al. 2014 fit an exponent of about 0.94. In 2D, the size of that tail depends on the
Modelled depth, for the same reason P_π does. Treat far-field crosstalk from a 2D model as an order-of-magnitude
estimate.

## Assumptions and limits of the 2D model

- **Infinitely long heater.** No heater ends, contacts or via spreading. P_π does not depend on the heater length in 2D.
  The length sets only R, V_π and the phase at a given power.
- **Linear.** k, ρc_p and dn/dT do not depend on temperature. The heater runs about 30 K above ambient at P_π, and Si's
  dn/dT changes by a few percent over that range.
- **Uniform Joule heating.** The current runs along the waveguide with uniform density in the TiN. There is no
  free-carrier or optical-loss change with temperature.
- **Substrate.** The Modelled depth is a model choice, not a converged parameter (see above). It shifts P_π by a few
  percent and sets the far-field crosstalk.
- **Inputs not read from the paper.** The thermal constants, heater thickness and cladding come from the two
  reproductions. The paper's full text could not be checked.

## Next

- [Thermal guide](../api/thermal.md): the setup, the numerical knobs and the results (`ThermalSim`, `ThermalResult`,
  `StepResponse`, `HeaterReport`), with links to the API reference.